In [5]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

POSTGRES_JAR = "/home/jovyan/work/spark/jars/postgresql-42.7.4.jar"
POSTGRES_URL = "jdbc:postgresql://postgres:5432/spark"

POSTGRES_PROPS = {
    "user": "postgres",
    "password": "postgres",
    "driver": "org.postgresql.Driver",
}

spark = (
    SparkSession.builder
    .appName("postgres-star-etl")
    .config("spark.jars", POSTGRES_JAR)
    .config("spark.driver.extraClassPath", POSTGRES_JAR)
    .config("spark.executor.extraClassPath", POSTGRES_JAR)
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

mock = spark.read.jdbc(
    url=POSTGRES_URL,
    table="mock_data",
    properties=POSTGRES_PROPS,
)

print("mock_data rows:", mock.count())
mock.printSchema()

mock_data rows: 10000
root
 |-- id: integer (nullable = true)
 |-- customer_first_name: string (nullable = true)
 |-- customer_last_name: string (nullable = true)
 |-- customer_age: integer (nullable = true)
 |-- customer_email: string (nullable = true)
 |-- customer_country: string (nullable = true)
 |-- customer_postal_code: string (nullable = true)
 |-- customer_pet_type: string (nullable = true)
 |-- customer_pet_name: string (nullable = true)
 |-- customer_pet_breed: string (nullable = true)
 |-- seller_first_name: string (nullable = true)
 |-- seller_last_name: string (nullable = true)
 |-- seller_email: string (nullable = true)
 |-- seller_country: string (nullable = true)
 |-- seller_postal_code: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- product_price: decimal(38,18) (nullable = true)
 |-- product_quantity: integer (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- sale_customer_id: strin

In [15]:
def nullif_empty(col_name):
    return F.when(F.col(col_name) == "", None).otherwise(F.col(col_name))


def add_id(df, id_col, order_cols):
    window = Window.orderBy(*[F.col(c).asc_nulls_last() for c in order_cols])
    return df.withColumn(id_col, F.row_number().over(window))

def ns_eq(left_col, right_col):
    return left_col.eqNullSafe(right_col)

In [7]:
dim_customer = (
    mock.select(
        nullif_empty("customer_first_name").alias("first_name"),
        nullif_empty("customer_last_name").alias("last_name"),
        F.col("customer_age").cast("int").alias("age"),
        nullif_empty("customer_email").alias("email"),
        nullif_empty("customer_country").alias("country"),
        nullif_empty("customer_postal_code").alias("postal_code"),
    )
    .dropDuplicates()
)

dim_customer = (
    add_id(
        dim_customer,
        "id",
        ["first_name", "last_name", "age", "email", "country", "postal_code"],
    )
    .select("id", "first_name", "last_name", "age", "email", "country", "postal_code")
)

print(dim_customer.count())
dim_customer.show(5, truncate=False)

10000
+---+----------+------------+---+--------------------------+------------+-----------+
|id |first_name|last_name   |age|email                     |country     |postal_code|
+---+----------+------------+---+--------------------------+------------+-----------+
|1  |Aaren     |Adriano     |55 |fdillistoneem@symantec.com|China       |NULL       |
|2  |Aaren     |Chipperfield|42 |jgiffordhf@wordpress.com  |Iraq        |NULL       |
|3  |Aaren     |Eagers      |44 |wjesson3r@patch.com       |Philippines |1106       |
|4  |Aaron     |Biggins     |82 |fslinnjj@ifeng.com        |South Africa|2661       |
|5  |Ab        |Denniston   |43 |fmixon76@cnet.com         |China       |NULL       |
+---+----------+------------+---+--------------------------+------------+-----------+
only showing top 5 rows



In [8]:
dim_pet = (
    mock.select(
        nullif_empty("pet_category").alias("category"),
        nullif_empty("customer_pet_type").alias("type"),
        nullif_empty("customer_pet_name").alias("name"),
        nullif_empty("customer_pet_breed").alias("breed"),
    )
    .dropDuplicates()
)

dim_pet = (
    add_id(
        dim_pet,
        "id",
        ["category", "type", "name", "breed"],
    )
    .select("id", "category", "type", "name", "breed")
)

print(dim_pet.count())
dim_pet.show(5, truncate=False)

9850
+---+--------+----+-------+--------+
|id |category|type|name   |breed   |
+---+--------+----+-------+--------+
|1  |Birds   |bird|Abbott |Parakeet|
|2  |Birds   |bird|Abigael|Siamese |
|3  |Birds   |bird|Abner  |Parakeet|
|4  |Birds   |bird|Addy   |Parakeet|
|5  |Birds   |bird|Adel   |Parakeet|
+---+--------+----+-------+--------+
only showing top 5 rows



In [9]:
dim_seller = (
    mock.select(
        nullif_empty("seller_first_name").alias("first_name"),
        nullif_empty("seller_last_name").alias("last_name"),
        nullif_empty("seller_email").alias("email"),
        nullif_empty("seller_country").alias("country"),
        nullif_empty("seller_postal_code").alias("postal_code"),
    )
    .dropDuplicates()
)

dim_seller = (
    add_id(
        dim_seller,
        "id",
        ["first_name", "last_name", "email", "country", "postal_code"],
    )
    .select("id", "first_name", "last_name", "email", "country", "postal_code")
)

print(dim_seller.count())
dim_seller.show(5, truncate=False)

10000
+---+----------+---------+-------------------------+-------------+-----------+
|id |first_name|last_name|email                    |country      |postal_code|
+---+----------+---------+-------------------------+-------------+-----------+
|1  |Aarika    |Brussell |abrussellad@time.com     |Indonesia    |NULL       |
|2  |Aaron     |Hemphrey |ahemphreym0@goodreads.com|Iceland      |225        |
|3  |Aaron     |Sheerin  |asheerinqc@mit.edu       |Indonesia    |NULL       |
|4  |Ab        |Copsey   |acopseyns@mail.ru        |Philippines  |4217       |
|5  |Abagael   |Charley  |acharleyh3@imdb.com      |New Caledonia|98828      |
+---+----------+---------+-------------------------+-------------+-----------+
only showing top 5 rows



In [10]:
dim_supplier = (
    mock.select(
        nullif_empty("supplier_name").alias("name"),
        nullif_empty("supplier_contact").alias("contact"),
        nullif_empty("supplier_email").alias("email"),
        nullif_empty("supplier_phone").alias("phone"),
        nullif_empty("supplier_address").alias("address"),
        nullif_empty("supplier_city").alias("city"),
        nullif_empty("supplier_country").alias("country"),
    )
    .dropDuplicates()
)

dim_supplier = (
    add_id(
        dim_supplier,
        "id",
        ["name", "contact", "email", "phone", "address", "city", "country"],
    )
    .select("id", "name", "contact", "email", "phone", "address", "city", "country")
)

print(dim_supplier.count())
dim_supplier.show(5, truncate=False)

10000
+---+-----+------------------+-------------------------+------------+------------+----------+-----------+
|id |name |contact           |email                    |phone       |address     |city      |country    |
+---+-----+------------------+-------------------------+------------+------------+----------+-----------+
|1  |Abata|Aldwin Clee       |acleejh@nyu.edu          |836-276-9778|Room 102    |Maglaj    |Malaysia   |
|2  |Abata|Amandi Sandars    |asandars54@google.nl     |523-840-0452|PO Box 69892|Tonshayevo|Japan      |
|3  |Abata|Bernadene Bossel  |bbossellv@fema.gov       |812-276-2743|PO Box 14734|Evansville|Australia  |
|4  |Abata|Branden Beverstock|bbeverstockpe@weather.com|719-212-1871|PO Box 60865|Xidianzi  |Poland     |
|5  |Abata|Clint Bovey       |cbovey67@wufoo.com       |566-551-4781|Apt 932     |Qesarya   |Netherlands|
+---+-----+------------------+-------------------------+------------+------------+----------+-----------+
only showing top 5 rows



In [11]:
dim_product = (
    mock.select(
        nullif_empty("product_name").alias("name"),
        nullif_empty("product_category").alias("category"),
        F.col("product_price").cast("decimal(10,2)").alias("price"),
        F.col("product_quantity").cast("int").alias("quantity"),
        F.col("product_weight").cast("decimal(10,1)").alias("weight"),
        nullif_empty("product_color").alias("color"),
        nullif_empty("product_size").alias("size"),
        nullif_empty("product_brand").alias("brand"),
        nullif_empty("product_material").alias("material"),
        nullif_empty("product_description").alias("description"),
        F.col("product_rating").cast("decimal(2,1)").alias("rating"),
        F.col("product_reviews").cast("int").alias("reviews"),
        F.to_date("product_release_date").alias("release_date"),
        F.to_date("product_expiry_date").alias("expiry_date"),
    )
    .dropDuplicates()
)

dim_product = (
    add_id(
        dim_product,
        "id",
        [
            "name", "category", "price", "quantity", "weight", "color", "size",
            "brand", "material", "description", "rating", "reviews",
            "release_date", "expiry_date"
        ],
    )
    .select(
        "id", "name", "category", "price", "quantity", "weight", "color", "size",
        "brand", "material", "description", "rating", "reviews",
        "release_date", "expiry_date"
    )
)

print(dim_product.count())
dim_product.show(5, truncate=False)

10000
+---+---------+--------+-----+--------+------+-------+------+----------+--------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------+-------+------------+-----------+
|id |name     |category|price|quantity|weight|color  |size  |brand     |material|description                                                                                                                                                                                                                                                                                 |rating|reviews|release_date|expiry_date|
+---+---------+--------+-----+--------+------+-------+------+----------+--------+---------------------------------------------------------------------------------------------------

In [12]:
dim_store = (
    mock.select(
        nullif_empty("store_name").alias("name"),
        nullif_empty("store_location").alias("location"),
        nullif_empty("store_city").alias("city"),
        nullif_empty("store_state").alias("state"),
        nullif_empty("store_country").alias("country"),
        nullif_empty("store_phone").alias("phone"),
        nullif_empty("store_email").alias("email"),
    )
    .dropDuplicates()
)

dim_store = (
    add_id(
        dim_store,
        "id",
        ["name", "location", "city", "state", "country", "phone", "email"],
    )
    .select("id", "name", "location", "city", "state", "country", "phone", "email")
)

print(dim_store.count())
dim_store.show(5, truncate=False)

10000
+---+-----+----------+------------+-----+----------+------------+--------------------------+
|id |name |location  |city        |state|country   |phone       |email                     |
+---+-----+----------+------------+-----+----------+------------+--------------------------+
|1  |Abata|11th Floor|Ibiúna      |NULL |Montenegro|660-676-5491|cchevisp5@npr.org         |
|2  |Abata|12th Floor|San Vicente |NULL |Argentina |230-952-8959|lgoodacregs@vinaora.com   |
|3  |Abata|14th Floor|Batasan Bata|NULL |China     |996-610-3702|pesbyg4@guardian.co.uk    |
|4  |Abata|14th Floor|Norsborg    |AB   |Portugal  |155-350-7386|shanshaw8h@sfgate.com     |
|5  |Abata|18th Floor|Cikupa      |NULL |Portugal  |940-737-0532|tpitkeathlykt@google.co.jp|
+---+-----+----------+------------+-----+----------+------------+--------------------------+
only showing top 5 rows



In [16]:
fact_source = (
    mock
    .withColumn("sale_quantity_clean", F.col("sale_quantity").cast("int"))
    .withColumn("sale_total_price_clean", F.col("sale_total_price").cast("decimal(10,2)"))
    .withColumn("sale_date_clean", F.to_date("sale_date"))
)

fact_sale = (
    fact_source.alias("m")

    .join(
        dim_customer.alias("c"),
        (
            ns_eq(nullif_empty("customer_first_name"), F.col("c.first_name")) &
            ns_eq(nullif_empty("customer_last_name"), F.col("c.last_name")) &
            ns_eq(F.col("m.customer_age").cast("int"), F.col("c.age")) &
            ns_eq(nullif_empty("customer_email"), F.col("c.email")) &
            ns_eq(nullif_empty("customer_country"), F.col("c.country")) &
            ns_eq(nullif_empty("customer_postal_code"), F.col("c.postal_code"))
        ),
        "left"
    )

    .join(
        dim_seller.alias("s"),
        (
            ns_eq(nullif_empty("seller_first_name"), F.col("s.first_name")) &
            ns_eq(nullif_empty("seller_last_name"), F.col("s.last_name")) &
            ns_eq(nullif_empty("seller_email"), F.col("s.email")) &
            ns_eq(nullif_empty("seller_country"), F.col("s.country")) &
            ns_eq(nullif_empty("seller_postal_code"), F.col("s.postal_code"))
        ),
        "left"
    )

    .join(
        dim_product.alias("pr"),
        (
            ns_eq(nullif_empty("product_name"), F.col("pr.name")) &
            ns_eq(nullif_empty("product_category"), F.col("pr.category")) &
            ns_eq(F.col("m.product_price").cast("decimal(10,2)"), F.col("pr.price")) &
            ns_eq(F.col("m.product_quantity").cast("int"), F.col("pr.quantity")) &
            ns_eq(F.col("m.product_weight").cast("decimal(10,1)"), F.col("pr.weight")) &
            ns_eq(nullif_empty("product_color"), F.col("pr.color")) &
            ns_eq(nullif_empty("product_size"), F.col("pr.size")) &
            ns_eq(nullif_empty("product_brand"), F.col("pr.brand")) &
            ns_eq(nullif_empty("product_material"), F.col("pr.material")) &
            ns_eq(nullif_empty("product_description"), F.col("pr.description")) &
            ns_eq(F.col("m.product_rating").cast("decimal(2,1)"), F.col("pr.rating")) &
            ns_eq(F.col("m.product_reviews").cast("int"), F.col("pr.reviews")) &
            ns_eq(F.to_date("product_release_date"), F.col("pr.release_date")) &
            ns_eq(F.to_date("product_expiry_date"), F.col("pr.expiry_date"))
        ),
        "left"
    )

    .join(
        dim_store.alias("st"),
        (
            ns_eq(nullif_empty("store_name"), F.col("st.name")) &
            ns_eq(nullif_empty("store_location"), F.col("st.location")) &
            ns_eq(nullif_empty("store_city"), F.col("st.city")) &
            ns_eq(nullif_empty("store_state"), F.col("st.state")) &
            ns_eq(nullif_empty("store_country"), F.col("st.country")) &
            ns_eq(nullif_empty("store_phone"), F.col("st.phone")) &
            ns_eq(nullif_empty("store_email"), F.col("st.email"))
        ),
        "left"
    )

    .join(
        dim_supplier.alias("su"),
        (
            ns_eq(nullif_empty("supplier_name"), F.col("su.name")) &
            ns_eq(nullif_empty("supplier_contact"), F.col("su.contact")) &
            ns_eq(nullif_empty("supplier_email"), F.col("su.email")) &
            ns_eq(nullif_empty("supplier_phone"), F.col("su.phone")) &
            ns_eq(nullif_empty("supplier_address"), F.col("su.address")) &
            ns_eq(nullif_empty("supplier_city"), F.col("su.city")) &
            ns_eq(nullif_empty("supplier_country"), F.col("su.country"))
        ),
        "left"
    )

    .join(
        dim_pet.alias("pe"),
        (
            ns_eq(nullif_empty("pet_category"), F.col("pe.category")) &
            ns_eq(nullif_empty("customer_pet_type"), F.col("pe.type")) &
            ns_eq(nullif_empty("customer_pet_name"), F.col("pe.name")) &
            ns_eq(nullif_empty("customer_pet_breed"), F.col("pe.breed"))
        ),
        "left"
    )

    .select(
        F.col("c.id").alias("customer_id"),
        F.col("s.id").alias("seller_id"),
        F.col("pr.id").alias("product_id"),
        F.col("st.id").alias("store_id"),
        F.col("su.id").alias("supplier_id"),
        F.col("pe.id").alias("pet_id"),
        F.col("m.sale_quantity_clean").alias("quantity"),
        F.col("m.sale_total_price_clean").alias("total_price"),
        F.col("m.sale_date_clean").alias("date"),
    )
)

fact_sale = (
    add_id(
        fact_sale,
        "id",
        [
            "date",
            "customer_id",
            "seller_id",
            "product_id",
            "store_id",
            "supplier_id",
            "pet_id",
            "quantity",
            "total_price",
        ],
    )
    .select(
        "id",
        "date",
        "customer_id",
        "seller_id",
        "product_id",
        "store_id",
        "supplier_id",
        "pet_id",
        "quantity",
        "total_price",
    )
)

print("fact_sale rows:", fact_sale.count())

fact_sale rows: 10000


In [17]:
fact_sale.select(
    F.sum(F.when(F.col("customer_id").isNull(), 1).otherwise(0)).alias("null_customer_id"),
    F.sum(F.when(F.col("pet_id").isNull(), 1).otherwise(0)).alias("null_pet_id"),
    F.sum(F.when(F.col("seller_id").isNull(), 1).otherwise(0)).alias("null_seller_id"),
    F.sum(F.when(F.col("supplier_id").isNull(), 1).otherwise(0)).alias("null_supplier_id"),
    F.sum(F.when(F.col("product_id").isNull(), 1).otherwise(0)).alias("null_product_id"),
    F.sum(F.when(F.col("store_id").isNull(), 1).otherwise(0)).alias("null_store_id"),
).show()

+----------------+-----------+--------------+----------------+---------------+-------------+
|null_customer_id|null_pet_id|null_seller_id|null_supplier_id|null_product_id|null_store_id|
+----------------+-----------+--------------+----------------+---------------+-------------+
|               0|          0|             0|               0|              0|            0|
+----------------+-----------+--------------+----------------+---------------+-------------+



In [ ]:
tables = {
    "dim_customer": dim_customer,
    "dim_pet": dim_pet,
    "dim_seller": dim_seller,
    "dim_supplier": dim_supplier,
    "dim_product": dim_product,
    "dim_store": dim_store,
    "fact_sale": fact_sale,
}

for table_name, df in tables.items():
    print(f"Writing {table_name}: {df.count()} rows")

    (
        df.write
        .mode("overwrite")
        .jdbc(
            url=POSTGRES_URL,
            table=table_name,
            properties=POSTGRES_PROPS,
        )
    )

print("Done")

Writing dim_customer: 10000 rows
Writing dim_pet: 9850 rows
Writing dim_seller: 10000 rows
Writing dim_supplier: 10000 rows
Writing dim_product: 10000 rows
Writing dim_store: 10000 rows
Writing fact_sale: 10000 rows
Done


In [19]:
spark.stop()